In [4]:
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState, add_messages
from typing import TypedDict, Annotated

from app.core.config import settings

from langchain.chat_models import init_chat_model

# 定初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={"thinking":{"type":"disabled"}}
)

class MessageAgentState(TypedDict):
    messages:Annotated[list[BaseMessage], add_messages]

def call_llm(state: MessagesState):
    ai_message = model.invoke(state["messages"])
    return {"messages":[ai_message]}

# 3.定义边，构建图
builder = StateGraph(MessagesState)

builder.add_node("llm", call_llm)

builder.add_edge(START, "llm")
builder.add_edge("llm", END)

graph = builder.compile(checkpointer=InMemorySaver())

config = {
    "configurable":{
        "thread_id":"1"
    }
}

# 4.调用Graph
result = graph.stream_events({
    "messages":[HumanMessage(content="给西游记续写一个开头,需要至少三千字")]
},
config, version="v3"
)

for message in result.messages:
    for chunk in message.text:
        print(chunk, end="")

## 西游记续写·第一章 泾河龙君再告状 大唐天子又犯愁
> 书接前文：唐僧师徒取经归来，五圣成真，三界同庆。然真经虽得，人心难测；功德虽圆，因果未消。自那唐僧回东土大唐，长安城里讲经三年，度化无数。谁料想，一场泼天官司，竟从九幽地府直闹到灵霄宝殿——只因当年那泾河老龙王，魂魄困在幽冥，日日喊冤，夜夜告状，哭得十殿阎君个个头疼。这一日，状纸竟递到了玉帝案前……

诗曰：

八十一难已西游，五圣功成万古秋。
谁料真经传东土，反教旧鬼闹新愁。
长安市上僧说偈，幽冥界里龙呼冤。
若要西天取经事，从头再问老君牛。

话说唐三藏师徒四人，自西天取经归来，功德圆满，如来佛祖各授封号：唐僧为旃檀功德佛，孙悟空为斗战胜佛，猪八戒为净坛使者，沙僧为金身罗汉，白马亦封八部天龙广力菩萨。师徒谢恩已毕，各奔前程。唐僧留在大唐长安，于慈恩寺中开坛讲经，弘扬佛法，超度亡魂。太宗皇帝亲赐袈裟锡杖，又拨银十万两修缮寺宇，一时香火鼎盛，长安城内外善男信女，排着队往慈恩寺里挤，挤得那山门前的石狮子都磨得锃亮。

这一日，正是贞观二十三年三月初三，上巳佳节。长安城里桃红柳绿，曲江池畔踏青的人摩肩接踵。唐僧正在大雄宝殿上讲《金刚经》，忽听半空中一阵阴风刮过，吹得殿角铜铃叮当乱响，供桌上的烛火“噗”地矮了三寸。众人只觉后脖颈发凉，抬头看时，殿外晴空万里，却不知这冷气从何而来。唐僧心里咯噔一下，手中念珠顿了顿，继续讲经不提。

你道这阴风从何而来？原来那泾河龙王，当年因与袁守诚赌赛，私改降雨点数，被魏征梦中斩首，一灵真性直坠幽冥。这龙君到了阴曹地府，越想越冤：我虽是龙王，却也是奉命行雨，只因赌了口气，犯了天条，可那袁守诚一个凡间术士，怎就晓得玉帝旨意？分明是有人与他通风报信！再者，魏征一介臣子，凭什么能在梦中斩龙？定是唐王李世民御口亲许，才让魏征下了这狠手。千错万错，错在那唐王言而无信！当年他答应救我，转头却让魏征砍了我脑袋，这笔账不算清楚，我泾河龙王死不瞑目！

于是这老龙魂魄，日日在森罗殿前击鼓鸣冤。十殿阎君被他吵得脑仁疼，判官翻烂了生死簿，发现这案子确实有蹊跷——袁守诚算卦、魏征梦斩、唐王应诺，环环相扣，像是有人刻意安排。秦广王皱眉道：“此事牵涉天庭命官、人界帝王，地府无权擅断。”楚江王摆手：“往上递！递到天曹去！”于是层层上报，那状纸一路从幽冥背阴山，经过南天门，最后竟摆到了玉皇大帝的御案上。

玉帝

In [5]:
response = graph.stream_events({
    "messages":[HumanMessage(content="用一段话总结都续写了什么")]
},
config, version="v3"
)
for message in response.messages:
    for chunk in message.text:
        print(chunk, end="")

这段续写以泾河龙王在幽冥界喊冤告状为引，写他状告唐太宗李世民当年言而无信、纵容魏征梦中斩龙，官司一路闹到玉帝案前；玉帝派太白金星下凡，先到长安慈恩寺找正在讲经的唐僧了解情况，并透露玉帝的解决方案——让李世民亲赴幽冥向龙王赔罪、请袁守诚后人做水陆道场化解因果、再由大唐出资为龙王修庙供香火以求来生福报；太白金星还建议让孙悟空先去“先兵后礼”地镇住性情执拗的龙王，随后猪八戒、沙僧也掺和进来，悟空当即一个筋斗奔赴幽冥找龙王算账，由此拉开新一轮因果纠葛的序幕。